In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset

# 1. Comprobar si hay GPU disponible
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de cómputo: {device}")

# 2. Cargar el dataset procesado
df = pd.read_csv('../data/processed/reviews_en_clean.csv')
df['review'] = df['review'].fillna("")         #Solo para asegurarme de que no haya valores nulos en la columna de texto, ya que el modelo no puede procesar valores nulos.
df['label'] = df['recommended'].astype(int)    #El motivo de esta nueva columna es que el modelo de HuggingFace requiere que la columna de etiquetas sea de tipo entero, no booleano, ademas de que la libreria Transformers busca una columna llamada 'label' por defecto para trabajar.

# 3. Train / Test Split (mismo random_state para consistencia con el baseline)
train_df, test_df = train_test_split(
    df, 
    test_size=0.20, 
    random_state=42, 
    stratify=df['label']
)

print(f"Total Entrenamiento: {len(train_df)} | Total Test: {len(test_df)}")

c:\My proyects\steam_review_analyzer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo de cómputo: cpu
Total Entrenamiento: 132952 | Total Test: 33239


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
import evaluate

# 1. Cargar el tokenizador preentrenado de DistilBERT
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 2. Tomar un subset estratificado representativo para entrenar en CPU
train_subset, _ = train_test_split(
    train_df,
    train_size=8000,
    random_state=42,
    stratify=train_df['label']
)

# 3. Convertir a formato Hugging Face Dataset, solo seleccionamos las dos columnas que va a utilizar el pipeline de Transformers, el resto se descartan
train_hf = Dataset.from_pandas(train_subset[['review', 'label']]) 
test_hf = Dataset.from_pandas(test_df[['review', 'label']])

# 4. Función de tokenización (límite fijo de 128 tokens con padding y truncado)
def tokenize_function(examples):
    return tokenizer(
        examples["review"], 
        padding="max_length", 
        truncation=True, 
        max_length=128
    )

print("Tokenizando conjuntos de datos...")
train_tokenized = train_hf.map(tokenize_function, batched=True)
test_tokenized = test_hf.map(tokenize_function, batched=True)

# 5. Definir la función para calcular Macro F1 durante la evaluación
metric_f1 = evaluate.load("f1")
metric_precision = evaluate.load("precision")
metric_recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    macro_f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
    precision = metric_precision.compute(predictions=predictions, references=labels, average="macro")["precision"]
    recall = metric_recall.compute(predictions=predictions, references=labels, average="macro")["recall"]
    
    return {
        "macro_f1": macro_f1,
        "macro_precision": precision,
        "macro_recall": recall
    }

print("¡Tokenización lista y métricas configuradas!")

c:\My proyects\steam_review_analyzer\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tahie\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokenizando conjuntos de datos...


Map: 100%|██████████| 33239/33239 [00:02<00:00, 15636.25 examples/s]

¡Tokenización lista y métricas configuradas!


In [ ]:
import torch
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight

# 1. Calcular los pesos de clase exactos sobre el subconjunto de entrenamiento
# Esto replica el comportamiento de class_weight='balanced' utilizado en modelos de sklearn, como la fase anterior, pero ahora aplicado a PyTorch.
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_subset['label']),
    y=train_subset['label'].values
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Pesos calculados -> Clase 0 (Negativo): {weights_tensor[0]:.2f} | Clase 1 (Positivo): {weights_tensor[1]:.2f}")

# 2. Instanciar DistilBERT para Clasificación Binaria (num_labels=2)
model = DistilBertForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=2
)
model.to(device)

# 3. Crear una subclase de Trainer que aplique CrossEntropyLoss ponderada, sobreescribiendo el método compute_loss para incorporar los pesos de clase calculados previamente.
class BalancedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Función de pérdida ponderada con los pesos de clase
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

Pesos calculados -> Clase 0 (Negativo): 20.30 | Clase 1 (Positivo): 0.51


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2483.09it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Interpretación de la salida anterior

#### 1. Pesos calculados
* **Clase 0 (Negativo):** `20.30`
* **Clase 1 (Positivo):** `0.51`
* **Fundamento:** Para neutralizar el sesgo dominante del 97.5% de reseñas positivas, el error (*loss*) derivado de clasificar erróneamente una reseña negativa recibe un multiplicador $\approx 40$ veces superior al de una reseña positiva, forzando al optimizador a ajustar gradientes significativos sobre la clase minoritaria.

---

#### 2. Mecanismo de Transfer Learning (`UNEXPECTED` vs. `MISSING`)
* **Parámetros `UNEXPECTED` (`vocab_projector`, `vocab_transform`, etc.):** Corresponden a la cabeza original de DistilBERT utilizada durante su preentrenamiento para tareas de lenguaje enmascarado (*Masked Language Modeling*). Se descartan de forma segura al no ser requeridas para clasificación de secuencias.
* **Parámetros `MISSING` (`classifier.*`, `pre_classifier.*`):** Representan la nueva cabeza de clasificación binaria (2 neuronas de salida) acoplada sobre el *backbone* del Transformer. Se inicializan con pesos aleatorios para ser entrenados y calibrados durante el proceso de *fine-tuning* que viene a continuacion.

In [4]:
from transformers import TrainingArguments

# 1. Hiperparámetros de entrenamiento adaptados para CPU
training_args = TrainingArguments(
    output_dir="./results_distilbert", # Carpeta donde se guardan checkpoints
    eval_strategy="epoch",             # Evaluar el modelo al final de cada época
    save_strategy="epoch",             # Guardar una copia del modelo por época
    learning_rate=2e-5,                # Tasa de aprendizaje pequeña para no destruir el conocimiento previo
    per_device_train_batch_size=32,    # Procesa 32 reseñas por paso de gradiente
    per_device_eval_batch_size=64,     # En evaluación no calcula gradientes, puede usar lotes más grandes
    num_train_epochs=2,                # 2 pasadas completas al conjunto de entrenamiento
    weight_decay=0.01,                 # Regularización L2 para evitar sobreajuste (overfitting)
    logging_steps=50,                  # Muestra métricas de pérdida cada 50 pasos
    load_best_model_at_end=True,       # Carga automáticamente la mejor versión al terminar
    metric_for_best_model="macro_f1",  # Criterio para elegir el mejor modelo
    greater_is_better=True,
    report_to="none"
)

# 2. Inicializar el Trainer balanceado
trainer = BalancedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics
)

# 3. Iniciar el entrenamiento
print("Iniciando Fine-Tuning de DistilBERT en CPU...")
trainer.train()

Iniciando Fine-Tuning de DistilBERT en CPU...


c:\My proyects\steam_review_analyzer\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Macro F1,Macro Precision,Macro Recall
1,0.555179,0.477775,0.720499,0.675248,0.804554
2,0.331887,0.715712,0.761640,0.762268,0.761016


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.17it/s]
c:\My proyects\steam_review_analyzer\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.36it/s]


TrainOutput(global_step=500, training_loss=0.5033493423461914, metrics={'train_runtime': 6715.3612, 'train_samples_per_second': 2.383, 'train_steps_per_second': 0.074, 'total_flos': 529869594624000.0, 'train_loss': 0.5033493423461914, 'epoch': 2.0})

In [5]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# 1. Obtener predicciones finales sobre el test set
predictions = trainer.predict(test_tokenized)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# 2. Classification Report detallado
print("--- REPORTE DE CLASIFICACIÓN (DISTILBERT) ---")
print(classification_report(labels, preds, target_names=["Negativo (0)", "Positivo (1)"], digits=2))

# 3. Matriz de Confusión
print("--- MATRIZ DE CONFUSIÓN ---")
print(confusion_matrix(labels, preds))

# 4. Guardar modelo y tokenizador
model_path = "../models/distilbert_sentiment"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)
print(f"\nModelo y tokenizador guardados exitosamente en: {model_path}")

c:\My proyects\steam_review_analyzer\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


--- REPORTE DE CLASIFICACIÓN (DISTILBERT) ---
              precision    recall  f1-score   support

Negativo (0)       0.54      0.53      0.53       817
Positivo (1)       0.99      0.99      0.99     32422

    accuracy                           0.98     33239
   macro avg       0.76      0.76      0.76     33239
weighted avg       0.98      0.98      0.98     33239

--- MATRIZ DE CONFUSIÓN ---
[[  436   381]
 [  377 32045]]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.18it/s]


Modelo y tokenizador guardados exitosamente en: ../models/distilbert_sentiment


# Fase 3: Clasificador con Deep Learning (DistilBERT Fine-Tuning)

## 1. Contexto y justificación arquitectónica
Para superar las limitaciones del modelo lineal (TF-IDF + Regresión Logística), el cual adolecía de una precisión deficiente en la clase minoritaria (0.24) y carecía de comprensión semántica sobre negaciones y sarcasmo, se implementó una arquitectura basada en Transformers: **`distilbert-base-uncased`**.

* **Ventajas:** Retiene el 97% del entendimiento contextual de BERT pero reduce un 40% el número de parámetros y acelera un 60% la inferencia.
* **Mecanismo:** Procesamiento bidireccional de secuencias mediante mecanismos de auto-atención (*Self-Attention*) con complejidad $\mathcal{O}(N^2)$.

---

## 2. Decisiones Técnicas y Optimizaciones para CPU

1. **Gestión de Longitud de Secuencia (`max_length=128`):**  
   Se aplicó truncado y padding a 128 tokens, reduciendo el espacio de atención cuadrática 16 veces respecto al límite teórico (512 tokens) sin pérdida de información relevante en las reseñas.
2. **Muestreo Estratificado para Entrenamiento:**  
   Se realizó el ajuste fino sobre una muestra balanceada de **8,000 ejemplos** (`stratify=y`) para viabilizar el entrenamiento local en CPU en ~1.8 horas, manteniendo la evaluación sobre el 100% del conjunto de prueba original (33,239 registros).
3. **Ponderación de Clases (`BalancedTrainer`):**  
   Para neutralizar el desbalance severo (97.5% positivo vs 2.5% negativo), se extendió el método `compute_loss` del `Trainer` incorporando una función `nn.CrossEntropyLoss` ponderada con pesos calculados analíticamente:
   * Clase 0 (Negativo): `20.30`
   * Clase 1 (Positivo): `0.51`

---

## 3. Resultados Finales y Comparativa

```text
--- REPORTE DE CLASIFICACIÓN (DISTILBERT) ---
              precision    recall  f1-score   support

Negativo (0)       0.54      0.53      0.53       817
Positivo (1)       0.99      0.99      0.99     32422

    accuracy                           0.98     33239
   macro avg       0.76      0.76      0.76     33239
weighted avg       0.98      0.98      0.98     33239

--- MATRIZ DE CONFUSIÓN ---
[[  436   381]
 [  377 32045]]
```

## 4. Oportunidades de mejora y trabajo futuro

Si bien el modelo actual superó ampliamente el baseline y proporciona una base sólida para inferencia local en CPU, existen líneas claras de optimización para futuras iteraciones:

1. **Escalabilidad de Datos (Entrenamiento Completo en GPU):**  
   * **Estado actual:** Se entrenó con un subconjunto de 8.000 muestras debido a las restricciones de cómputo en CPU local.
   * **Propuesta:** Utilizar aceleración por hardware (GPU) para procesar el 100% del conjunto de entrenamiento (~132.000 registros), exponiendo al modelo a una mayor diversidad léxica y jerga de la comunidad gamer.

2. **Calibración del Umbral de Decisión (*Threshold Tuning*):**  
   * **Estado actual:** La clasificación opera con `argmax` (umbral estático $P \ge 0.50$).
   * **Propuesta:** Ajustar la frontera de decisión sobre las probabilidades calibradas para ponderar dinámicamente el compromiso entre Precision y Recall según los requerimientos operativos de la plataforma.

3. **Exploración de Arquitecturas Avanzadas:**  
   * Evaluar arquitecturas como **RoBERTa-base** o **DeBERTa-v3**, las cuales ofrecen mecanismos de atención desacoplada y mejor rendimiento en la captura de matices sintácticos complejos y sarcasmo.